In [7]:
import sys; sys.path.insert(0, "../src")
from protgen.models.transformer_rocha import Transformer

In [8]:
import torch
from torch import nn
import torch.nn.functional as F

In [9]:
import torch.optim as optim

In [10]:
import matplotlib.pyplot as plt

In [11]:
from protgen.data import ProteinDataset
from torch.utils.data import DataLoader

train_data = ProteinDataset(path_to_data="../data/train.csv")
val_data = ProteinDataset(path_to_data="../data/val.csv")
test_data = ProteinDataset(path_to_data="../data/test.csv")

print(f"Data class: {train_data}")

# Get tensors
X_train = train_data.tokens
y_train_labels = train_data.labels          # classification task
y_train_shifted = train_data.tokens_shifted # generative task

print(f"# Train data: \n Tokens shape: {X_train.shape}\n Labels (shape): {y_train_labels.shape} \n Shifted shape: {y_train_shifted.shape}")


# Inddexing
x, x_shifted, y, length = train_data[0]
print(f"# Sample 0: \nTokens: \n {x}\n Shifted: \n {x_shifted}\n Label: {y}\n Length: {length}")

# DataLoader
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

Data class: ProteinDataset(num_samples=16020, vocab_size=23, num_classes=3, max_length=129)
# Train data: 
 Tokens shape: torch.Size([16020, 129])
 Labels (shape): torch.Size([16020]) 
 Shifted shape: torch.Size([16020, 129])
# Sample 0: 
Tokens: 
 tensor([ 1, 13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,
         3, 19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6,
        22, 16,  8, 11, 19, 10, 14, 12, 11,  9, 18,  3, 11, 18, 10, 13,  5, 10,
        13, 18, 12, 18, 10, 17, 15,  8, 19, 11,  7, 22, 10, 11,  3,  6,  8, 10,
         5,  6, 20, 16,  3, 10, 16, 12, 10, 16,  5, 22, 12, 22, 11,  6, 11, 12,
        10, 14, 22,  2,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0,  0])
 Shifted: 
 tensor([13, 10, 11, 11, 16,  7, 14, 10, 10, 11, 14, 19,  8,  7,  3, 17, 15,  3,
        19, 12, 12, 20, 11, 12, 18, 16, 17,  7, 19, 18,  5, 10, 18, 12,  6, 22

In [12]:
val_loader = DataLoader(val_data, batch_size=32, shuffle=True)

In [14]:
transformer = Transformer(num_heads=4, num_layers=2, vocabulary_size=23, embedding_dim=32)

c:\Users\marti\OneDrive\Documents\GitHub\02456-autoregressive-project\notebooks\../src\protgen\models\transformer_rocha.py:41: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(encoder_layers, norm=nn.LayerNorm(embedding_dim), num_layers=num_layers)


In [15]:
print(list(transformer.named_parameters()))

[('embedding.weight', Parameter containing:
tensor([[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 1.2330e+00, -1.3709e+00,  6.2306e-01, -1.0456e+00, -4.0772e-01,
          4.9130e-01, -1.2881e-01, -5.7024e-01,  5.5614e-01,  8.8544e-01,
         -4.4723e-01,  8.5087e-01, -2.0700e-01,  1.5451e+00, -3.4361e-01,
         -3.0576e-01, -7.3713e-01,  1.0737e+00,  1.2067e+00,  1.9934e-01,
         -6.8785e-01, -4.3769e-01,  1.8063e+00,  6.1884e-01, -1.5141e-01,
          2.0187e+00,  1.9969e+00, -1.8873e-01, -1.7459e-01,  1.3553e-01,
         -1.1583e+00,  5.0407e-0

In [16]:
train_data.df

,sequence,family
0,MIKKQFNIIKNTGFARPATLLVKLSQRFTSDISLEYQGKTINLKHS...,HPr
1,MNKHELITRISEKSGMTKKDVESVLNGFMDEVTDALGSGNKVQLIG...,HU
2,MTQRFEDTFLIQNKLGLHARAASVLAELSAQFDANITLFQGSKEAE...,HPr
3,MNKTELILRIAEETGFSRKNVELGLNAAINLITKALTEEEKVQIVG...,HU
4,MVERKAVIKNEAGIHCRPTAVITQEAAKVDAVIMILSPGGTCQLGS...,HPr
...,...,...
16015,MTKADIINEIANETGVAKKDVTVVVESFMATIKDSLLEKRECVYLR...,HU
16016,MSRLSRQVTIVNQKGLHARASAKFVTFVSRLPEGLKVEVEKDGQCV...,HPr
16017,MNKSDLINHVAAETGLTKADSGFALEAVLEGITKSLRKGDTVTLTG...,HU
16018,MTLTKAELADLLFEKVGLNKREAKDMVEAFFEEIRNALERGEGVKL...,HU


In [17]:
val_data.df

,sequence,family
0,MNKADFVKHVAQQHQCTQIEAEKTIDMFTSSVIDAMGKGNEISLVG...,HU
1,MSERKATIASRVGLHARPASIFAAAVGELPIEVTIGLEADPVEDAM...,HPr
2,MNKSQLIDKIAAGAELSKADAGRALDAFIGAVTDALKEGDSVALVG...,HU
3,MNKSELVDKMAEKSGLSKKDSDKALNAMLESVEEALTSGDKVQLIG...,HU
4,MNKTELIDHIASKADISKAAAGRTLDALIGAVKTTLKKGGTVTLVG...,HU
...,...,...
1996,MDNRSRREAPEAVDTHGYYKAEIEVTHGFGLHARPSVIFTRLAKSF...,HPr
1997,MNKSELIDAIAASADIPKAVAGRALDAVIDSVTGALKAGDSVVLVG...,HU
1998,MSEHESAGRKLVSVDFEIFGYVQGVCFRMYTEREGLRLGLVGWVKN...,AcyP
1999,MNKSELIDAIAESADISKAAAGRALDAMTNTITGALKNDDQVTLIG...,HU


In [18]:
test_data.df

,sequence,family
0,MTKAEVIVAIADKTGVDKASVSTTLESFFEIVKDTLAQGDAVYVRG...,HU
1,MERVTAVARGRVQGVGYRYYVSECAHATGIHGWVKNEPDGTVRIVA...,AcyP
2,MPQSSRTVLIQNRLGLHARAAVKLVELAQSFEAVLTIQNQEGKEAT...,HPr
3,MKHLNLKIHGKVQGVFFREKTKQLAQELGLNGYVKNTSDNCVEVVV...,AcyP
4,MTDRTRAHVYVSGKVQGVYFRATTREEAREQGVDGWVRNLDDGRVE...,AcyP
...,...,...
1996,MNKSELIQAVAERSELTKKDAGSAIQAAFDIISETLEKGETVQLIG...,HU
1997,MANKAELIDSVASKTGLTKKDATSAVDAVFETIQDNLSEGNKVQLI...,HU
1998,MPKLERQVTICNKLGLHARAATKLVVLASEFDASITLIQGEKQASA...,HPr
1999,MAAASAGGPSDPPTPQQSSPTKAVRVVVKGRVQGVFFRDWTVETAR...,AcyP


***CHECK WINDOWING IS WORKING - Transformer should only be able to look at the previous sequence of the token it is predicting***

In [ ]:
transformer(torch.tensor([[1, 2, 3, 4, 5]]))